# 08 Maps
Generate maps into `outputs/maps` with boundary outlines.


In [ ]:
import sys
import importlib
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.utils import load_config, ensure_dir
import src.maps as maps_mod
importlib.reload(maps_mod)

print('python =', sys.executable)

missing = []
try:
    import geopandas as _gpd
    print('geopandas =', _gpd.__version__)
except Exception:
    missing.append('geopandas')

try:
    import shapefile as _pyshp
    print('pyshp =', _pyshp.__version__)
except Exception:
    missing.append('pyshp')

if missing:
    raise RuntimeError('Missing libs for boundary map: ' + ', '.join(missing) + '. Please install from requirements.txt')

cfg = load_config('../config/config.yaml')
cfg.setdefault('map_output', {})
cfg['map_output']['out_subdir'] = 'maps'
cfg['map_output']['draw_boundary'] = True

boundary_path = cfg['map_output'].get('boundary_path')
if not boundary_path or (not Path(boundary_path).exists()):
    raise RuntimeError('boundary_path is missing or not found in config/config.yaml')

OUT = ensure_dir('../outputs')
p1 = OUT / 'clean_data.parquet'
assert p1.exists(), f'missing: {p1}'
df = pd.read_parquet(p1)

print('maps module =', maps_mod.__file__)
print('output root =', OUT.resolve())
print('target out_subdir =', cfg['map_output']['out_subdir'])
print('draw_boundary =', cfg['map_output']['draw_boundary'])
print('boundary_path =', boundary_path)

outs = maps_mod.make_maps(df, cfg, OUT)
print('n_maps =', len(outs))
if outs:
    print('first =', outs[0])
    print('final dir =', (OUT / cfg['map_output']['out_subdir']).resolve())
